
# 🧪 Hybrid Text Features (Pro): **TF‑IDF ⊕ Doc2Vec** → XGBoost — with DBOW Hybrid, Stratified K‑Fold CV & SHAP

Models compared on BBC News:
1. **TF‑IDF → XGBoost**
2. **Doc2Vec DM → XGBoost** (PV‑DM)
3. **Doc2Vec DBOW → XGBoost** (PV‑DBOW)
4. **Hybrid (TF‑IDF ⊕ Doc2Vec‑DM) → XGBoost**
5. **Hybrid‑DBOW (TF‑IDF ⊕ Doc2Vec‑DBOW) → XGBoost**

Extras:
- **Stratified K‑Fold CV** (default K=5) with per‑model mean/±std of accuracy, precision, recall, F1 (weighted)
- **Hyperparameter search** for the best Hybrid (DM) model
- **Feature importance** (top n‑grams & hybrid dims)
- **Optional SHAP** explanations for the tuned Hybrid (DM) model (auto‑skips if `shap` not installed)


## 0) Setup

In [ ]:

# If in Colab, uncomment to install dependencies:
# !pip -q install gensim xgboost scikit-learn matplotlib tqdm shap

import os, re, string, multiprocessing
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
from tqdm import tqdm

import xgboost as xgb
from gensim.models.doc2vec import TaggedDocument, Doc2Vec

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Try SHAP (optional)
try:
    import shap
    SHAP_AVAILABLE = True
except Exception:
    SHAP_AVAILABLE = False
    print("SHAP not available; SHAP section will be skipped. Install `shap` to enable.")


## 1) Load BBC dataset

In [ ]:

CSV_PATHS = ['/mnt/data/bbc-text.csv', './bbc-text.csv']

csv_path = None
for p in CSV_PATHS:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    try:
        from google.colab import files  # type: ignore
        print("Upload 'bbc-text.csv' (two columns: category,text).")
        uploaded = files.upload()
        csv_path = list(uploaded.keys())[0]
    except Exception as e:
        raise FileNotFoundError("Could not find 'bbc-text.csv'. Please place it in the working directory or upload it.") from e

df = pd.read_csv(csv_path)
df.head()


## 2) Minimal Cleaning

In [ ]:

def clean_text(s: str) -> str:
    s = s.lower()
    s = s.translate(str.maketrans('', '', string.punctuation))
    s = re.sub(r'\d+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


## 3) Doc2Vec transformer (DM/DBOW)

In [ ]:

class Doc2VecTransformer:
    def __init__(self, vector_size=200, epochs=20, learning_rate=0.02, min_alpha=0.0001, alpha_decay=0.95,
                 dm=1, window=5, min_count=2, workers=None, seed=42):
        self.vector_size = vector_size
        self.epochs = epochs
        self.learning_rate = float(learning_rate)
        self.min_alpha = float(min_alpha)
        self.alpha_decay = float(alpha_decay)
        self.dm = dm
        self.window = window
        self.min_count = min_count
        self.workers = (multiprocessing.cpu_count() - 1) if workers is None else int(workers)
        self.seed = seed
        self.model = None

    def _tagged(self, texts):
        return [TaggedDocument(clean_text(t).split(), [i]) for i, t in enumerate(texts)]

    def fit(self, texts, y=None):
        tagged = self._tagged(texts)
        model = Doc2Vec(
            vector_size=self.vector_size,
            dm=self.dm, window=self.window,
            min_count=self.min_count, workers=self.workers,
            seed=self.seed
        )
        model.build_vocab(tagged)

        alpha = self.learning_rate
        for _ in tqdm(range(self.epochs), desc=f"Doc2Vec training (dm={self.dm})"):
            shuffled = tagged.copy()
            rng = np.random.RandomState(self.seed)
            rng.shuffle(shuffled)
            model.train(shuffled, total_examples=len(shuffled), epochs=1, start_alpha=alpha, end_alpha=alpha)
            alpha = max(self.min_alpha, alpha * self.alpha_decay)
            model.alpha = alpha; model.min_alpha = alpha

        self.model = model
        return self

    def transform(self, texts):
        assert self.model is not None, "Call fit() before transform()."
        vecs = [self.model.infer_vector(clean_text(t).split(), alpha=self.min_alpha, steps=20, random_seed=self.seed) for t in texts]
        return np.asarray(vecs)

    def fit_transform(self, texts, y=None):
        return self.fit(texts, y).transform(texts)


## 4) Build feature sets: TF‑IDF, Doc2Vec (DM & DBOW), and Hybrids

In [ ]:

y = df['category']

# TF-IDF
tfidf = TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1,2))
X_tfidf = tfidf.fit_transform(df['text'].map(clean_text))

# Doc2Vec DM
d2v_dm = Doc2VecTransformer(vector_size=200, epochs=20, dm=1)
X_dm = d2v_dm.fit_transform(df['text'])

# Doc2Vec DBOW
d2v_dbow = Doc2VecTransformer(vector_size=200, epochs=20, dm=0)
X_dbow = d2v_dbow.fit_transform(df['text'])

# Hybrids
from scipy.sparse import csr_matrix, hstack
X_hybrid_dm   = hstack([X_tfidf, csr_matrix(X_dm)],   format='csr')
X_hybrid_dbow = hstack([X_tfidf, csr_matrix(X_dbow)], format='csr')

X_tfidf.shape, X_dm.shape, X_dbow.shape, X_hybrid_dm.shape, X_hybrid_dbow.shape


## 5) Train/Test split (holdout for confusion matrices)

In [ ]:

Xtf_tr, Xtf_te, y_tr, y_te = train_test_split(X_tfidf,   y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
Xdm_tr, Xdm_te, _, _       = train_test_split(X_dm,      y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
Xdb_tr, Xdb_te, _, _       = train_test_split(X_dbow,    y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
Xhy_tr, Xhy_te, _, _       = train_test_split(X_hybrid_dm,   y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
Xhyb_tr, Xhyb_te, _, _     = train_test_split(X_hybrid_dbow, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

Xtf_tr.shape, Xdm_tr.shape, Xdb_tr.shape, Xhy_tr.shape, Xhyb_tr.shape


## 6) Train baseline XGBoost + tune Hybrid (DM)

In [ ]:

def train_xgb(X_train, y_train):
    clf = xgb.XGBClassifier(
        n_estimators=600, max_depth=6, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9,
        objective='multi:softprob', eval_metric='mlogloss',
        tree_method='hist', random_state=RANDOM_STATE
    )
    clf.fit(X_train, y_train)
    return clf

def search_xgb(X_train, y_train):
    base = xgb.XGBClassifier(
        objective='multi:softprob', eval_metric='mlogloss',
        tree_method='hist', random_state=RANDOM_STATE
    )
    param_grid = {
        'n_estimators': [300, 600],
        'max_depth': [5, 7],
        'learning_rate': [0.05, 0.1],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    }
    gs = GridSearchCV(base, param_grid=param_grid, cv=3, n_jobs=-1, scoring='f1_weighted', verbose=1)
    gs.fit(X_train, y_train)
    print("Best params:", gs.best_params_)
    print("Best CV f1_weighted:", gs.best_score_)
    return gs.best_estimator_

m_tfidf_clf  = train_xgb(Xtf_tr, y_tr)
m_dm_clf     = train_xgb(Xdm_tr, y_tr)
m_dbow_clf   = train_xgb(Xdb_tr, y_tr)
m_hybrid_clf = search_xgb(Xhy_tr, y_tr)  # tuned hybrid (DM)


## 7) Evaluate on holdout (accuracy, precision, recall, F1, confusion matrix)

In [ ]:

def evaluate_model(model, X_test, y_test, title="Model"):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

    print(f"\n=== {title} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pr:.4f}  Recall: {rc:.4f}  F1: {f1:.4f}\n")
    print(classification_report(y_test, y_pred, zero_division=0))

    plt.figure(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, xticks_rotation='vertical')
    plt.title(f"Confusion Matrix — {title}")
    plt.tight_layout(); plt.show()

    return {"accuracy": acc, "precision_w": pr, "recall_w": rc, "f1_w": f1}

r_tfidf  = evaluate_model(m_tfidf_clf,  Xtf_te, y_te, title="XGB (TF-IDF)")
r_dm     = evaluate_model(m_dm_clf,     Xdm_te, y_te, title="XGB (Doc2Vec DM)")
r_dbow   = evaluate_model(m_dbow_clf,   Xdb_te, y_te, title="XGB (Doc2Vec DBOW)")
r_hybrid = evaluate_model(m_hybrid_clf, Xhy_te, y_te, title="XGB (Hybrid TF-IDF ⊕ Doc2Vec-DM, tuned)")


## 8) Stratified K‑Fold CV (K=5) across all models

In [ ]:

def cv_scores(model_builder, X, y, K=5):
    skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=RANDOM_STATE)
    accs, prs, rcs, f1s = [], [], [], []
    for tr_idx, te_idx in skf.split(X, y):
        Xtr, Xte = X[tr_idx], X[te_idx]
        ytr, yte = y.iloc[tr_idx], y.iloc[te_idx]
        model = model_builder()
        model.fit(Xtr, ytr)
        yhat = model.predict(Xte)
        accs.append(accuracy_score(yte, yhat))
        pr, rc, f1, _ = precision_recall_fscore_support(yte, yhat, average='weighted', zero_division=0)
        prs.append(pr); rcs.append(rc); f1s.append(f1)
    return {
        "acc_mean": np.mean(accs), "acc_std": np.std(accs),
        "pr_mean": np.mean(prs),   "pr_std": np.std(prs),
        "rc_mean": np.mean(rcs),   "rc_std": np.std(rcs),
        "f1_mean": np.mean(f1s),   "f1_std": np.std(f1s),
    }

def make_tfidf(): return xgb.XGBClassifier(n_estimators=600, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9,
                                           objective='multi:softprob', eval_metric='mlogloss', tree_method='hist', random_state=RANDOM_STATE)
def make_dm():    return xgb.XGBClassifier(n_estimators=600, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9,
                                           objective='multi:softprob', eval_metric='mlogloss', tree_method='hist', random_state=RANDOM_STATE)
def make_dbow():  return xgb.XGBClassifier(n_estimators=600, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9,
                                           objective='multi:softprob', eval_metric='mlogloss', tree_method='hist', random_state=RANDOM_STATE)
def make_hybrid():return xgb.XGBClassifier(**m_hybrid_clf.get_params())  # reuse tuned params

cv_tfidf  = cv_scores(make_tfidf,  X_tfidf,     y)
cv_dm     = cv_scores(make_dm,     X_dm,        y)
cv_dbow   = cv_scores(make_dbow,   X_dbow,      y)
cv_hybrid = cv_scores(make_hybrid, X_hybrid_dm, y)

cv_table = pd.DataFrame([
    {"model":"TF-IDF", **cv_tfidf},
    {"model":"Doc2Vec DM", **cv_dm},
    {"model":"Doc2Vec DBOW", **cv_dbow},
    {"model":"Hybrid (DM, tuned)", **cv_hybrid},
]).set_index("model").round(4)

display(cv_table)


## 9) Feature importance and top tokens

In [ ]:

def top_features_from_xgb(model, feature_names, topk=25):
    importances = model.feature_importances_
    idx = np.argsort(importances)[::-1][:topk]
    return pd.DataFrame({"feature": np.array(feature_names)[idx], "importance": importances[idx]})

# Names for TF-IDF and Hybrid (DM)
tf_vocab = tfidf.get_feature_names_out()
dm_dim_names = [f"d2v_dm_{i}" for i in range(X_dm.shape[1])]
hy_dm_names = np.concatenate([tf_vocab, np.array(dm_dim_names)])

top_tfidf = top_features_from_xgb(m_tfidf_clf, tf_vocab, topk=25)
top_hy    = top_features_from_xgb(m_hybrid_clf, hy_dm_names, topk=30)

print("Top TF-IDF n-grams:"); display(top_tfidf)
plt.figure(figsize=(8,6))
plt.barh(np.arange(len(top_tfidf))[::-1], top_tfidf['importance'][::-1])
plt.yticks(np.arange(len(top_tfidf))[::-1], top_tfidf['feature'][::-1])
plt.title("Top 25 TF-IDF n-grams by Importance"); plt.tight_layout(); plt.show()

print("Top Hybrid features (n-grams + d2v_dm dims):"); display(top_hy)
plt.figure(figsize=(8,6))
plt.barh(np.arange(len(top_hy))[::-1], top_hy['importance'][::-1])
plt.yticks(np.arange(len(top_hy))[::-1], top_hy['feature'][::-1])
plt.title("Top 30 Hybrid Features by Importance"); plt.tight_layout(); plt.show()


## 10) (Optional) SHAP explanations for the tuned Hybrid (DM)

In [ ]:

if SHAP_AVAILABLE:
    # Use a small sample for speed
    n_sample = min(400, Xhy_te.shape[0])
    idx = np.random.choice(Xhy_te.shape[0], size=n_sample, replace=False)
    X_sample = Xhy_te[idx]
    explainer = shap.TreeExplainer(m_hybrid_clf)
    shap_values = explainer.shap_values(X_sample)
    # Summary plot
    shap.summary_plot(shap_values, X_sample, show=True)
else:
    print("SHAP not installed; skipping.")
